# 01. Importing Libraries

In [1]:
import os
import math
import random
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import scipy
import numpy as np
import pandas as pd
import scipy.stats as st

In [2]:
import sys
sys.path.append('..')

from src.functions import *

# 02. Importing data

In [3]:
places = pd.read_csv('../01. Data/01. Raw Data/brazil_tourism_intelligence.csv')

In [4]:
places.head()

,state,state_code,destination,type,best_months,good_months,avoid_months,why_best,category,international_draw,weather,crowd_level,description
0,Maranhão,MA,Lençóis Maranhenses,Nature / Ecotourism,June|July|August,May|September,January|February|March,Rainy season ends but lagoons still full — ide...,Nature / Ecotourism,High,hot,moderate,Vast desert landscape of white sand dunes fill...
1,Mato Grosso do Sul,MS,Pantanal,Wildlife / Ecotourism,July|August|September|October,April|May|June,November|December|January|February|March,Dry season — water levels low so wildlife conc...,Wildlife / Ecotourism,Very High,hot,low,The world's largest tropical wetland — a UNESC...
2,Amazonas,AM,Amazon Rainforest (dry season),Nature / Ecotourism,June|July|August|September,April|May,NaN,Dry season — easier trekking and wildlife spot...,Nature / Ecotourism,Very High,hot,low,"The world's largest rainforest, covering 60% o..."
3,Amazonas,AM,Amazon Rainforest (wet season),Nature / Ecotourism,January|February|March,November|December,NaN,Wet season — easier navigation by boat. Flora ...,Nature / Ecotourism,High,hot,low,Flooded forest season — boats navigate between...
4,Pernambuco,PE,Fernando de Noronha (diving),Marine / Diving,August|September|October|November,December|January,March|April|May,Clearest waters for diving (up to 50m visibili...,Marine / Diving,Very High,warm,moderate,Brazil's most exclusive island destination — a...


In [5]:
places.shape

(51, 13)

# 03. Data discovery, cleaning and wrangling

In [6]:
places.isnull().sum()

state                  0
state_code             0
destination            0
type                   0
best_months            0
good_months           11
avoid_months          20
why_best               0
category               0
international_draw     0
weather                0
crowd_level            0
description            0
dtype: int64

In [7]:
places.duplicated().sum()

np.int64(0)

In [8]:
places.dtypes

state                 str
state_code            str
destination           str
type                  str
best_months           str
good_months           str
avoid_months          str
why_best              str
category              str
international_draw    str
weather               str
crowd_level           str
description           str
dtype: object

In [9]:
# checking unique values in each column and understanding the content of the dataset
for column in places.columns:
    print(f'\n{"="*40}')
    print(f'{column} ({places[column].nunique()} unique values):')
    print(places[column].unique())


state (26 unique values):
<ArrowStringArray>
[           'Maranhão',  'Mato Grosso do Sul',            'Amazonas',
          'Pernambuco',      'Rio de Janeiro',           'São Paulo',
               'Bahia',               'Ceará',              'Paraná',
        'Minas Gerais', 'Rio Grande do Norte',                'Pará',
      'Santa Catarina',   'Rio Grande do Sul',           'Tocantins',
               'Piauí',         'Mato Grosso',             'Paraíba',
             'Alagoas',             'Sergipe',      'Espírito Santo',
               'Goiás',             'Roraima',                'Acre',
            'Rondônia',               'Amapá']
Length: 26, dtype: str

state_code (26 unique values):
<ArrowStringArray>
['MA', 'MS', 'AM', 'PE', 'RJ', 'SP', 'BA', 'CE', 'PR', 'MG', 'RN', 'PA', 'SC',
 'RS', 'TO', 'PI', 'MT', 'PB', 'AL', 'SE', 'ES', 'GO', 'RR', 'AC', 'RO', 'AP']
Length: 26, dtype: str

destination (51 unique values):
<ArrowStringArray>
[                         'Lençóis Maran

In [10]:
# drpping columns that seem to be duplicated
places.drop(columns=['category'], inplace=True)

In [11]:
# Renaming columns for better handling
places.rename(columns={'type': 'tourism_type'}, inplace=True)

In [12]:
# checking all lines for discrepancies in data (except why best and description, which will be checked separately)
places.drop(columns=['why_best', 'description'])

,state,state_code,destination,tourism_type,best_months,good_months,avoid_months,international_draw,weather,crowd_level
0,Maranhão,MA,Lençóis Maranhenses,Nature / Ecotourism,June|July|August,May|September,January|February|March,High,hot,moderate
1,Mato Grosso do Sul,MS,Pantanal,Wildlife / Ecotourism,July|August|September|October,April|May|June,November|December|January|February|March,Very High,hot,low
2,Amazonas,AM,Amazon Rainforest (dry season),Nature / Ecotourism,June|July|August|September,April|May,NaN,Very High,hot,low
3,Amazonas,AM,Amazon Rainforest (wet season),Nature / Ecotourism,January|February|March,November|December,NaN,High,hot,low
4,Pernambuco,PE,Fernando de Noronha (diving),Marine / Diving,August|September|October|November,December|January,March|April|May,Very High,warm,moderate
5,Pernambuco,PE,Fernando de Noronha (surfing),Surfing,January|February,NaN,NaN,High,warm,moderate
6,Rio de Janeiro,RJ,Rio de Janeiro (city),City / Culture / Beach,April|May|September|October|November,March|June|July,January|February,Very High,warm,high
7,Rio de Janeiro,RJ,Rio de Janeiro — Carnaval,Cultural Event,February|March,NaN,NaN,Very High,warm,very high
8,Rio de Janeiro,RJ,Rio de Janeiro — Réveillon,Cultural Event,December|January,NaN,NaN,Very High,warm,very high
9,Rio de Janeiro,RJ,Ilha Grande,Island / Beach,October|November|March|April,September|May,June|July|August,High,warm,moderate


In [13]:
# checking destinations, why best and descriptions make sense
for i, row in places.iterrows():
    print(f"--- {i} | {row['destination']} ---")
    print(f"WHY BEST: {row['why_best']}")
    print(f"DESCRIPTION: {row['description']}")
    print()

--- 0 | Lençóis Maranhenses ---
WHY BEST: Rainy season ends but lagoons still full — ideal for swimming and sandboarding
DESCRIPTION: Vast desert landscape of white sand dunes filled with crystal-clear freshwater lagoons after the rainy season. A surreal and unique ecosystem found nowhere else on Earth.

--- 1 | Pantanal ---
WHY BEST: Dry season — water levels low so wildlife concentrates. Best jaguar spotting Jul-Oct
DESCRIPTION: The world's largest tropical wetland — a UNESCO World Heritage Site and the best place on Earth to spot jaguars in the wild. Also home to giant otters, capybaras, caimans and hundreds of bird species.

--- 2 | Amazon Rainforest (dry season) ---
WHY BEST: Dry season — easier trekking and wildlife spotting. Beaches appear on riverbanks
DESCRIPTION: The world's largest rainforest, covering 60% of Brazil. Dry season offers easier trekking, river beaches and exceptional wildlife spotting. Base yourself in Manaus for access to jungle lodges.

--- 3 | Amazon Rainfor

In [14]:
# making the necessary changes to the description column to make it more readable and useful for the app
places.loc[2, 'why_best'] = "The world's largest rainforest. Dry season offers easier trekking, river beaches and exceptional wildlife spotting. Base yourself in Manaus for access to jungle lodges"

places.loc[3, 'description'] = "Flooded forest season — boats navigate between treetops in the iconic igapó (flooded forest). Pink river dolphins (locally known as Boto Cor-de-rosa) are more visible and the landscape is otherworldly."

places.loc[4, 'description'] = "Brazil's most exclusive island destination — a UNESCO marine reserve 350km offshore with the clearest waters in the South Atlantic. World-class diving with spinner dolphins, sea turtles and sharks. The island limits visitor numbers to protect the ecosystem."

places.loc[7, 'description'] = "The world's biggest carnival — five days of samba school parades at the Sambódromo plus street parties (blocos) across the city, filled with music and dance and vibrant costumes. An unmissable cultural spectacle."

places.loc[8, 'description'] = "The world's largest New Year's Eve celebration on Copacabana beach — over 2 million people in white gather for fireworks, live music and offerings to Iemanjá, the Afro-Brazilian goddess of the sea."

places.loc[9, 'why_best'] = "Warm water and less rain"

places.loc[10, 'why_best'] = "Warm summer months"

places.loc[11, 'why_best'] = "Mild weather"

places.loc[12, 'why_best'] = "Major international sports event"

places.loc[13, 'why_best'] = "Winter months — cool mountain air. Festival de Inverno in July features music, dance and theatre performances in the historic city centre"

places.loc[16, 'why_best'] = " Warm and sunny weather, ideal for beach activities"

places.loc[19, 'why_best'] = "Dry season — sunny and breezy"

places.loc[34, 'description'] = "The second largest festival in Brazil after Carnaval — a spectacular folkloric competition between two rival groups, each representing a folkloric bull: Garantido (red bull) and Caprichoso (blue bull). The competition is held on a floating arena in the Amazon river city of Parintins and draws visitors from across Brazil and abroad."

In [15]:
# checking the changes in random idexes
places.loc[11]

state                                                         São Paulo
state_code                                                           SP
destination                                            São Paulo (city)
tourism_type                                  City / Business / Culture
best_months                      March|April|September|October|November
good_months                                                  May|August
avoid_months                                      January|February|July
why_best                                                   Mild weather
international_draw                                            Very High
weather                                                            mild
crowd_level                                                        high
description           South America's largest city and Brazil's cult...
Name: 11, dtype: str

In [16]:
#creating new columns with category for app questions
category_map = {
    'Nature / Ecotourism': 'nature',
    'Wildlife / Ecotourism': 'nature',
    'Trekking / Nature': 'nature',
    'Waterfall / Nature': 'nature',
    'River Beach / Nature': 'nature',
    'Nature / Beach': 'nature',
    'Marine / Diving': 'beach',
    'Surfing': 'beach',
    'Island / Beach': 'beach',
    'Beach': 'beach',
    'Beach / Kitesurf': 'beach',
    'City / Beach': 'beach',
    'Beach / City': 'beach',
    'City / Culture / Beach': 'culture',
    'Cultural Event': 'culture',
    'City / Business / Culture': 'city',
    'Mountain / Culture': 'culture',
    'Colonial / Culture': 'culture',
    'Cultural / Religious Event': 'culture',
    'Archaeological': 'culture',
    'Sports Event': 'adventure',
    'Nature / Trekking': 'adventure',
    'City / Culture': 'culture',
}

places['category'] = places['tourism_type'].map(category_map)

In [17]:
# checking new values in the category column
places['category'].unique()

<ArrowStringArray>
['nature', 'beach', 'culture', 'city', 'adventure']
Length: 5, dtype: str

In [18]:
# making the month columsn more readable and consistent for the app
places['best_months'] = places['best_months'].apply(format_months)
places['good_months'] = places['good_months'].apply(format_months)
places['avoid_months'] = places['avoid_months'].apply(format_months)

In [21]:
# checking the changes in the columns
for column in ['best_months', 'good_months', 'avoid_months']:
    print(f'\n{"="*40}')
    print(f'{column} ({places[column].nunique()} unique values):')
    print(places[column].unique())


best_months (27 unique values):
<ArrowStringArray>
[                                                      'June | July | August',
                                        'July | August | September | October',
                                           'June | July | August | September',
                                                 'January | February | March',
                                    'August | September | October | November',
                                                         'January | February',
                               'April | May | September | October | November',
                                                           'February | March',
                                                         'December | January',
                                         'October | November | March | April',
                 'October | November | December | January | February | March',
                             'March | April | September | October | November',


In [23]:
places.head()

,state,state_code,destination,tourism_type,best_months,good_months,avoid_months,why_best,international_draw,weather,crowd_level,description,category
0,Maranhão,MA,Lençóis Maranhenses,Nature / Ecotourism,June | July | August,May | September,January | February | March,Rainy season ends but lagoons still full — ide...,High,hot,moderate,Vast desert landscape of white sand dunes fill...,nature
1,Mato Grosso do Sul,MS,Pantanal,Wildlife / Ecotourism,July | August | September | October,April | May | June,November | December | January | February | March,Dry season — water levels low so wildlife conc...,Very High,hot,low,The world's largest tropical wetland — a UNESC...,nature
2,Amazonas,AM,Amazon Rainforest (dry season),Nature / Ecotourism,June | July | August | September,April | May,,The world's largest rainforest. Dry season off...,Very High,hot,low,"The world's largest rainforest, covering 60% o...",nature
3,Amazonas,AM,Amazon Rainforest (wet season),Nature / Ecotourism,January | February | March,November | December,,Wet season — easier navigation by boat. Flora ...,High,hot,low,Flooded forest season — boats navigate between...,nature
4,Pernambuco,PE,Fernando de Noronha (diving),Marine / Diving,August | September | October | November,December | January,March | April | May,Clearest waters for diving (up to 50m visibili...,Very High,warm,moderate,Brazil's most exclusive island destination — a...,beach


# 04. Exporting data ready for use in app

In [22]:
places.to_pickle('../01. Data/02. Processed Data/intelligence_ready.pkl')
places.to_csv('../01. Data/02. Processed Data/intelligence_ready.csv', index=False)